# 0.环境配置

In [ ]:
import torch 
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from typing import Dict, List
import json


print(torch.cuda.is_available())

# 模型参数文件路径
model_file_path = "./model/qwen" 

# 设置模型加载的设备，cuda:0 为使用第一个显卡资源。device="cpu" 为调用CPU资源进行训练
device = "cuda:0"

# 系统提示词，根据训练任务的不同需要有不同的系统提示词
system_prompt = "你是一个煤矿安全领域的专家，请帮我生成风险分析报告"

# 训练数据的文件路径
data_file_path = r".\data\analysis_data.json"

# 每条训练数据的最大长度，分词器会将一个中文字切分为多个token，因此需要放开一些最大长度，保证数据的完整性
MAX_LENGTH = 2048

# LoRA微调后的模型文件输出路径
output_dir = r".\model\qwen-lora"

True


# 1.加载分词器

In [2]:
tokenizer = AutoTokenizer.from_pretrained(model_file_path, trust_remote_code=True)
tokenizer

Qwen2Tokenizer(name_or_path='/root/autodl-tmp/model/qwen', vocab_size=151643, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151647: AddedToken("<|object_ref_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151648: AddedToken("<|box_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151649: AddedToken("<|box_end|>", rstrip=F

# 2.加载训练集

In [3]:
def load_dataset(system_prompt: str, data_file_path: str, tokenizer: AutoTokenizer = None) -> Dataset:
    """
    从json文件中读取数据

    :param system_prompt: 构建数据集时给定的系统提示词
    :param data_file_path: json文件的路径
    :param tokenizer: 模型的分词器，可选传参
    :return: Huggingface的 Dataset对象
    """
    # 1. 手动读取 JSON 文件
    with open(data_file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # 构建符合格式的数据集
    data_format = []

    for d in data["conversations"]:
        system_content = ""
        # 用户的问题
        user_content = d[0]["value"]
        # llm的回答
        assistant_content = d[1]["value"]
        if system_prompt:
            # 系统提示词
            system_content = system_prompt
            data_format.append(
                [
                    {"role": "system", "content": system_content},
                    {"role": "user", "content": user_content},
                    {"role": "assistant", "content": assistant_content}
                ])
        else:
            data_format.append(
                [
                    {"role": "user", "content": user_content},
                    {"role": "assistant", "content": assistant_content}
                ])
    # 将数据转换为Hugging Face Dataset格式
    dataset = Dataset.from_dict({
        'conversations': data_format
    })

    return dataset

    
def process_func(example: Dict[str, List]):
    """
    遍历dataset对象，将每条数据进行按照Qwen的输入格式整理、分词、掩码等操作

    :param example: 每条数据的system、user、assistant文本
    :return: 转化、编码后的数据
    """
    example = example["conversations"]

    MAX_LENGTH = 2048  # Llama分词器会将一个中文字切分为多个token，因此需要放开一些最大长度，保证数据的完整性
    input_ids, attention_mask, labels = [], [], []
    instruction = tokenizer(
        f"<|im_start|>system\n{example[0]['content']}<|im_end|>\n<|im_start|>user\n{example[1]['content']}<|im_end|>\n<|im_start|>assistant\n",
        add_special_tokens=False)  # add_special_tokens 不在开头加 special_tokens
    response = tokenizer(f"{example[2]['content']}", add_special_tokens=False)
    input_ids = instruction["input_ids"] + response["input_ids"] + [tokenizer.pad_token_id]
    attention_mask = instruction["attention_mask"] + response["attention_mask"] + [1]  # 因为eos token咱们也是要关注的所以 补充为1
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"] + [tokenizer.pad_token_id]
    if len(input_ids) > MAX_LENGTH:  # 做一个截断
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }


In [ ]:
train_dataset = load_dataset(system_prompt=system_prompt,
                             data_file_path=data_file_path,
                             tokenizer=tokenizer)

# 编码后的数据集
train_dataset_tokenized = train_dataset.map(process_func, remove_columns=train_dataset.column_names)

print(tokenizer.decode(train_dataset_tokenized[0]['input_ids']))
print(tokenizer.decode(list(filter(lambda x: x != -100, train_dataset_tokenized[1]["labels"]))))

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

<|im_start|>system
你是一个煤矿安全领域的专家，请帮我生成风险分析报告<|im_end|>
<|im_start|>user
2020年4月4日，新疆兵团铁沟煤矿主斜井，施工单位擅自改变电缆下放方式且未制定操作规程，属违章指挥。<|im_end|>
<|im_start|>assistant
【分析报告】.时间：2020年4月4日.地点：新疆兵团铁沟煤矿主斜井.风险类别：运输风险.主要有害因素：违章指挥；擅自改变施工方案；未制定操作规程.导致风险的诱因：人为因素：管理人员擅自改变电缆下放方式，采用铁丝、尼龙绳将电缆和提升钢丝绳捆绑一体；未制定作业规程和操作规程；违章指挥作业。机器因素：使用木质杠杆刹车控制电缆下放速度，制动能力不足；954米长、重约7.6吨电缆失控加速下滑。环境因素：主斜井倾角25度，电缆从顺水沟送入，与水沟底板摩擦受损。管理因素：项目部未制定作业规程；未按规定进行技术交底；未对施工方案变更进行审批。<|endoftext|>
【分析报告】.时间：2020年4月4日.地点：新疆兵团铁沟煤矿主斜井.风险类别：运输风险.主要有害因素：临时组建队伍；人员未经培训；无证上岗.导致风险的诱因：人为因素：随意从社会上招收人员临时组建安装队；作业人员缺乏安全意识和操作技能；下井作业前未接受安全教育培训。管理因素：违反《煤矿整体托管安全管理办法（试行）》；未向项目部派驻整建制施工队伍；未建立人员培训档案和考核机制。<|endoftext|>


# 3.加载模型

In [5]:
model = AutoModelForCausalLM.from_pretrained(model_file_path,  # 模型参数文件路径
                                             device_map=device)  # 将模型加载到显卡上

model.enable_input_require_grads()  # 开启梯度检查点时，要执行该方法

model

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (ro

# 4. 配置LoRA 超参数

In [6]:
config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,  # 指定任务类型
    r=32,  # LoRa秩（秩越大，微调容量越大，同时资源消耗也更多）
    lora_alpha=16,  # 用于控制LoRA 更新模型参数的强度
    inference_mode=False,  # 训练模式
    target_modules=[
        "q_proj",
        "v_proj",
    ],  # 目标层
    lora_dropout=0.1,  # 丢弃率
    bias="none",  # 偏置项
)
model = get_peft_model(model, config)
# 打印LoRA 更新的参数量
model.print_trainable_parameters()

trainable params: 10,092,544 || all params: 7,625,709,056 || trainable%: 0.1323


# 5.配置训练参数并开始训练

In [ ]:
args = TrainingArguments(
    output_dir=output_dir,  # 训练后的模型参数保存文件路径
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    warmup_steps=5,
    num_train_epochs=8,  # 改这里
    learning_rate=2e-4,
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    save_steps=100,
    report_to="none",
)

# 开始训练
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset_tokenized,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

trainer.train()

Step,Training Loss
1,2.165812
2,2.368530
3,2.295334
4,2.120428
5,2.194882
6,2.322992
7,2.124744
8,1.887450
9,1.919729
10,1.810375


TrainOutput(global_step=1000, training_loss=0.3492027938812971, metrics={'train_runtime': 1416.9237, 'train_samples_per_second': 11.292, 'train_steps_per_second': 0.706, 'total_flos': 1.357355435347968e+17, 'train_loss': 0.3492027938812971, 'epoch': 8.0})